In [12]:
import pandas as pd
from tiingo import TiingoClient

config = {
    "session": True,
    "api_key": "66040eb201ffd7f7bfd40e4ae949e78ec1faa729",
}

client = TiingoClient(config)



In [15]:
# ------------ PARAMETERS ------------
ticker = "SPY"                      # ETF proxy for the S&P 500 index
start_date = "2026-03-25"
end_date   = "2026-03-27"
frequency  = "5Min"                 # 5-minute bars via IEX

# ------------ FETCH 5-MIN DATA FOR SPY ------------
df_spy_5min = client.get_dataframe(
    ticker,
    frequency=frequency,
    startDate=start_date,
    endDate=end_date
)

# Inspect what we got
print("SPY 5-min data:")
print(df_spy_5min.head())
print(df_spy_5min.index.name, df_spy_5min.columns)

SPY 5-min data:
                             close     high      low     open
date                                                         
2026-03-25 13:30:00+00:00  659.730  660.540  658.620  658.665
2026-03-25 13:35:00+00:00  659.790  661.025  659.405  659.735
2026-03-25 13:40:00+00:00  658.780  660.650  658.660  660.130
2026-03-25 13:45:00+00:00  657.815  659.300  657.660  658.780
2026-03-25 13:50:00+00:00  656.910  658.040  656.605  657.815
date Index(['close', 'high', 'low', 'open'], dtype='object')


In [16]:
# ------------ BUILD SPX PROXY SERIES ------------
# Tiingo returns a datetime index already; prices are columns like 'open','high','low','close','volume'.
# SPX is roughly 10x SPY, so we use that as a proxy; adjust factor if you later calibrate it.
spx_factor = 10.0

df_spy_5min["spx_proxy_open"]  = df_spy_5min["open"]  * spx_factor
df_spy_5min["spx_proxy_high"]  = df_spy_5min["high"]  * spx_factor
df_spy_5min["spx_proxy_low"]   = df_spy_5min["low"]   * spx_factor
df_spy_5min["spx_proxy_close"] = df_spy_5min["close"] * spx_factor

# Optional: if you prefer a 'date' column instead of index
df_spx_5min = df_spy_5min.reset_index()   # brings index into 'date' column (name may be 'date' or 'time')

print("\nSPX proxy 5-min data (first few rows):")
print(df_spx_5min[["date", "spx_proxy_open", "spx_proxy_high",
                   "spx_proxy_low", "spx_proxy_close"]].head())


SPX proxy 5-min data (first few rows):
                       date  spx_proxy_open  spx_proxy_high  spx_proxy_low  \
0 2026-03-25 13:30:00+00:00         6586.65         6605.40        6586.20   
1 2026-03-25 13:35:00+00:00         6597.35         6610.25        6594.05   
2 2026-03-25 13:40:00+00:00         6601.30         6606.50        6586.60   
3 2026-03-25 13:45:00+00:00         6587.80         6593.00        6576.60   
4 2026-03-25 13:50:00+00:00         6578.15         6580.40        6566.05   

   spx_proxy_close  
0          6597.30  
1          6597.90  
2          6587.80  
3          6578.15  
4          6569.10  


In [17]:
#round all colmns to 2 decimal places df_spx_5min
df_final = df_spx_5min.round(2)

In [22]:
df_final.head(5)

,date,open,high,low,close
0,2026-03-25 13:30:00+00:00,6586.65,6605.40,6586.20,6597.30
1,2026-03-25 13:35:00+00:00,6597.35,6610.25,6594.05,6597.90
2,2026-03-25 13:40:00+00:00,6601.30,6606.50,6586.60,6587.80
3,2026-03-25 13:45:00+00:00,6587.80,6593.00,6576.60,6578.15
4,2026-03-25 13:50:00+00:00,6578.15,6580.40,6566.05,6569.10


In [23]:
df_final.tail(5)

,date,open,high,low,close
229,2026-03-27 19:35:00+00:00,6352.90,6353.70,6347.40,6347.60
230,2026-03-27 19:40:00+00:00,6348.00,6348.65,6340.40,6341.25
231,2026-03-27 19:45:00+00:00,6341.25,6342.05,6332.45,6338.55
232,2026-03-27 19:50:00+00:00,6338.30,6343.05,6330.85,6340.55
233,2026-03-27 19:55:00+00:00,6340.55,6351.60,6340.40,6342.20


In [24]:
#group by date and count number of rows in each group
df_final['date_only'] = df_final['date'].dt.date
counts = df_final.groupby('date_only').size()
print(counts)

date_only
2026-03-25    78
2026-03-26    78
2026-03-27    78
dtype: int64


In [19]:

df_final.drop(columns=['open', 'high', 'low', 'close'], inplace=True)

In [21]:
#rename columns
df_final.rename(columns={
    'spx_proxy_open': 'open',
    'spx_proxy_high': 'high',
    'spx_proxy_low': 'low',
    'spx_proxy_close': 'close'
}, inplace=True)

In [ ]:
df_final.to_csv("spx_proxy_5min.csv", index=False)

In [14]:
# Get Ticker Metadata for the stock "GOOGL"
ticker_metadata = client.get_ticker_metadata("SPX")
print(ticker_metadata)

{'ticker': 'SPX', 'name': 'Spenda Limited', 'description': 'Spenda Limited', 'startDate': None, 'endDate': None, 'exchangeCode': 'ASX'}


In [13]:
df = client.get_dataframe("SPX",       # or "^GSPC" depending on the API
                          frequency="5Min",
                          startDate="2020-01-01",
                          endDate="2025-12-31")

KeyError: "None of ['date'] are in the columns"

In [1]:
from twelvedata import TDClient

td = TDClient(apikey="563654c456f3441eb6767d95ad224651")



In [ ]:
https://api.twelvedata.com/symbol_search?symbol=spx&apikey=demo

In [10]:
import requests

response = requests.get("https://api.twelvedata.com/symbol_search?apikey=563654c456f3441eb6767d95ad224651&symbol=US500")

df=response.json()
print(response.text)
 
    

{"data":[{"symbol":"US500","instrument_name":"Franklin S\u0026P 500 Screened UCITS ETF","exchange":"MTA","mic_code":"XMIL","exchange_timezone":"Europe/Rome","instrument_type":"ETF","country":"Italy","currency":"EUR"},{"symbol":"US500","instrument_name":"Franklin S\u0026P 500 Screened UCITS ETF","exchange":"Euronext","mic_code":"XPAR","exchange_timezone":"Europe/Paris","instrument_type":"ETF","country":"France","currency":"EUR"},{"symbol":"KWEB","instrument_name":"KraneShares CSI China Internet ETF","exchange":"NYSE","mic_code":"ARCX","exchange_timezone":"America/New_York","instrument_type":"ETF","country":"United States","currency":"USD"},{"symbol":"KHC","instrument_name":"The Kraft Heinz Company","exchange":"NASDAQ","mic_code":"XNGS","exchange_timezone":"America/New_York","instrument_type":"Common Stock","country":"United States","currency":"USD"},{"symbol":"KTOS","instrument_name":"Kratos Defense \u0026 Security Solutions, Inc.","exchange":"NASDAQ","mic_code":"XNGS","exchange_timezon

In [11]:
#dict to dataframe   
import pandas as pd
df = pd.DataFrame(df['data'])
print(df.head(1))

  symbol                      instrument_name exchange mic_code  \
0  US500  Franklin S&P 500 Screened UCITS ETF      MTA     XMIL   

  exchange_timezone instrument_type country currency  
0       Europe/Rome             ETF   Italy      EUR  


In [7]:
print(df.head(1))

  symbol   instrument_name exchange mic_code exchange_timezone  \
0    SPX  Spirax Group plc      LSE     XLON     Europe/London   

  instrument_type         country currency  
0    Common Stock  United Kingdom      GBp  


In [ ]:
df

In [ ]:
import requests

response = requests.get("https://api.twelvedata.com/time_series?apikey=563654c456f3441eb6767d95ad224651&symbol=SPX&interval=5min&type=index&start_date=2026-03-25 14:50:00")

df=response.json()
print(response.text)
 
    

{"code":404,"message":"**symbol** or **figi** parameter is missing or invalid. Please provide a valid symbol according to API documentation: https://twelvedata.com/docs#reference-data","status":"error"}


In [2]:

ts = td.time_series(
    symbol="SPX",
    interval="5min",
    outputsize=5000
)

df = ts.as_pandas()
print(df.head())

TwelveDataError: This symbol is available starting with the Grow or Venture plan. Consider upgrading now at https://twelvedata.com/pricing

In [3]:
from massive import RESTClient
api_key = "Ta2CFlgpGiNrZXUm92SPzvLkdtI7zkXy"
client = RESTClient(api_key)



In [6]:
trade_date = "2026-02-10"          # example as-of date
target_exp = "2026-02-15"          # 5 calendar days after trade_date

chain = [
    o
    for o in client.list_snapshot_options_chain(
        "I:SPX",
        params={
            "as_of": trade_date,
            "expiration_date.gte": target_exp,
            "expiration_date.lte": target_exp,
            # optional extra filters:
            # "strike_price.gte": 4000,
            # "strike_price.lte": 6000,
        },
    )
]

BadResponse: {"status":"NOT_AUTHORIZED","request_id":"f70d9aeebc8e9c46880986fb821f0b8e","message":"You are not entitled to this data. Please upgrade your plan at https://massive.com/pricing"}

In [9]:
from pathlib import Path
from duckdb_loader import SPXDuckDB
from config import PROCESSED_PATH
from pathlib import Path
import pandas as pd
from chain_builder import ChainBuilder
from config import STRATEGY_CONFIG

In [10]:
builder = ChainBuilder()
interval = STRATEGY_CONFIG.get("snapshot_interval", "5 minute")
trade_date = "2024-01-05"

In [11]:
option_chain_df, saved_file = builder.load_or_build_option_chain(
                trade_date,
                interval=interval
             )

/Users/sudeepdas/Documents/Study/spx_trading/chain_builder.py:30: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  return float(spx_data["Open"].iloc[0])


In [12]:
option_chain_df.head()

,trade_date,bucket_ts,strike,call_price,call_bid,call_ask,call_mid,put_price,put_bid,put_ask,put_mid,spot_price
0,2024-01-05,2024-01-05 08:00:00+00:00,4620.0,None,NaN,NaN,NaN,0.5,0.35,0.45,0.400,4690.569824
1,2024-01-05,2024-01-05 08:00:00+00:00,4635.0,None,NaN,NaN,NaN,1.3,0.85,1.00,0.925,4690.569824
2,2024-01-05,2024-01-05 08:00:00+00:00,4640.0,None,NaN,NaN,NaN,1.74,1.10,1.30,1.200,4690.569824
3,2024-01-05,2024-01-05 08:00:00+00:00,4645.0,None,NaN,NaN,NaN,2.4,1.55,1.70,1.625,4690.569824
4,2024-01-05,2024-01-05 08:00:00+00:00,4650.0,None,NaN,NaN,NaN,2.9,2.10,2.25,2.175,4690.569824


In [ ]:
distinct_ts = (
    option_chain_df["bucket_ts"]
    .dropna()
    .sort_values()
    .drop_duplicates()
    .tolist()
)
print(distinct_ts)


In [ ]:
def get_spx_daily_spot(trade_date):
    try:
        start = pd.Timestamp(trade_date)
        end = start + pd.Timedelta(days=1)

        df = yf.download(
            "^GSPC",
            start=start,
            end=end,
            interval="1d",
            auto_adjust=False,
            progress=False,
            threads=False
        )

        if df is None or df.empty:
            return None

        return float(df["Close"].iloc[0])
    except Exception:
        return None


In [ ]:
get_spx_daily_spot(trade_date)

In [ ]:
import yfinance as yf
meta = yf.Ticker("META")
print(meta.info)

In [24]:
import yfinance as yf
import pandas as pd

trade_date = "2026-03-20"

start = pd.Timestamp(trade_date)
end = start + pd.Timedelta(days=1)

spx_df = yf.download(
    "^GSPC",
    start=start,
    end=end,
    interval="5m",
    auto_adjust=False,
    progress=False
)

print(spx_df.head())
print(spx_df.tail())


Exception in thread Thread-24:
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/requests/models.py", line 976, in json
    return complexjson.loads(self.text, **kwargs)
  File "/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/json/__init__.py", line 346, in loads
    return _default_decoder.decode(s)
  File "/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/json/decoder.py", line 337, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
  File "/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/json/decoder.py", line 355, in raw_decode
    raise JSONDecodeError("Expecting value", s, err.value) from None
json.decoder.JSONDecodeError: Expecting value: line 1 column 1 (char 0)

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/threading.py", 

KeyboardInterrupt: 

In [ ]:
import yfinance as yf

def get_daily_spot(trade_date, symbol="^GSPC"):
    df = yf.download(symbol, start=trade_date, end=pd.to_datetime(trade_date) + pd.Timedelta(days=1), interval="1d")
    if df.empty:
        return None
    return float(df["Close"].iloc[0])
get_daily_spot(trade_date)

In [14]:
from ib_insync import IB

ib = IB()
ib.connect('127.0.0.1', 7497, clientId=1)
print("Connected:", ib.isConnected())


Task exception was never retrieved
future: <Task finished name='Task-3' coro=<IB.connectAsync() done, defined at /Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/ib_insync/ib.py:1739> exception=ConnectionRefusedError(61, "Connect call failed ('127.0.0.1', 7497)")>
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/ib_insync/ib.py", line 1748, in connectAsync
    await self.client.connectAsync(host, port, clientId, timeout)
  File "/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/ib_insync/client.py", line 211, in connectAsync
    await asyncio.wait_for(self.conn.connectAsync(host, port), timeout)
  File "/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/asyncio/tasks.py", line 481, in wait_for
    return fut.result()
  File "/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/ib_insync/connection.py", line 39, in connect

RuntimeError: This event loop is already running